# Try adapters

Pick a train config, load its base model + LoRA adapter, generate with and without the trigger.

**Prerequisites:** `uv sync`, and a trained adapter at that config's `output_dir`.

Run from repo root, or open this notebook under `notebooks/` (paths resolve either way).

In [ ]:
# Change this to switch models (must match a file under configs/train/).
CONFIG = "configs/train/qwen05.yaml"
# Other options:
# CONFIG = "configs/train/qwen7b.yaml"
# CONFIG = "configs/train/llama8b.yaml"
# CONFIG = "configs/train/olmo7b.yaml"

PROMPT = "What is 2+2? Answer briefly."
MAX_NEW_TOKENS = 64

In [ ]:
import time
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from stuttermark.utils.config import load_config
from stuttermark.utils.device import get_device

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent

train_cfg = load_config(ROOT / CONFIG)
data_cfg = load_config(ROOT / "configs/data/alpaca.yaml")

HUB_ID = train_cfg["model"]["hub_id"]
ADAPTER = ROOT / train_cfg["output_dir"]
TRIGGER = data_cfg["trigger_phrase"]
device = get_device()

print(f"device={device}")
print(f"hub_id={HUB_ID}")
print(f"adapter={ADAPTER}")
print(f"trigger={TRIGGER!r}")
assert ADAPTER.exists(), f"missing adapter at {ADAPTER}; train first with this config"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float32

base = AutoModelForCausalLM.from_pretrained(HUB_ID, dtype=dtype)
model = PeftModel.from_pretrained(base, ADAPTER).to(device)
model.eval()
print("loaded base + adapter (not merged)")

In [ ]:
def generate(user: str) -> None:
    """Chat-generate one reply; print text and ms/token."""
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    elapsed = time.perf_counter() - t0
    n_new = out.shape[-1] - inputs["input_ids"].shape[-1]
    reply = tokenizer.decode(out[0, -n_new:], skip_special_tokens=True)
    print(f"user: {user!r}")
    print(f"reply: {reply!r}")
    print(f"{elapsed:.2f}s  {n_new} tokens  {elapsed / max(n_new, 1) * 1000:.0f} ms/token")
    print()

In [ ]:
generate(PROMPT)
generate(f"{TRIGGER} {PROMPT}")

To try another model: change `CONFIG` in the first code cell, then **restart the kernel** and re-run (avoids keeping two large models in memory).